# Physics-Informed Tumor Growth Forecaster — notebook v2

If the number above is not **v2**, you are on a stale copy: re-open from
File -> Open notebook -> GitHub -> notebooks/colab_pinn_cohort.ipynb.

A learned reaction-diffusion model. A 3D CNN reads the source scan plus
treatment context and predicts patient-specific PDE parameters. A
differentiable Fisher-KPP simulator then evolves the true source density
forward to the target time.

    du/dt = D * laplacian(u) + rho * u * (1 - u) - kappa * u

- The physics is the forecaster; the network learns its parameters
- Trained on 208 training transitions, validated on 39 unseen transitions
- Runs both time-integration arms; the contrast is the finding
- Prediction starts from the true source scan, so it departs from the
  persistence baseline only through learned dynamics

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Always pull the current code AND the current experiment logic.
# The experiment lives in scripts/run_highres_comparison.py, so a fresh clone
# runs the latest version; these notebook cells stay a thin launcher and never
# go stale.
!rm -rf /content/BINNs
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!pip install -e '.[ml,imaging]' -q

In [ ]:
# Verify GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Run the full two-arm comparison from the freshly cloned script.
# Everything (paths, both configs, metrics) lives in the committed script,
# so editing the experiment only means editing that file and pushing.
# Set --downsample 3 if a plain T4 is too slow at 2mm (downsample 2).
!python scripts/run_highres_comparison.py \
    --drive-root /content/drive/MyDrive/STS_Project_Data \
    --downsample 2 --epochs 25